In [19]:
from numpy import select
import pandas as pd
import duckdb
pd.set_option('display.max_columns', None)

chemin_fichier = r"retailcorps.parquet"

df=duckdb.query(f"""
    select *
    From read_parquet('{chemin_fichier}')
""").df()

In [20]:
df.head(5)

,OrderLineID,Invoice,StockCode,Description,Quantity,InvoiceDate,CustomerID,Country,ProductCategoryCode,ProductCategory,ProductFamily,ProductNameClean,UnitPriceGBP,UnitCostGBP,CustomerSegment,CustomerType,LoyaltyProgram,SalesChannel,PaymentMethod,WarehouseID,PromotionCode,DiscountRate,OrderStatus,DeliveryTime,Continent,Region,Currency
0,1,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,13085.0,United Kingdom,SEASONAL,Seasonal,Occasions,15Cm Christmas Glass Ball 20 Lights,17.82,8.47,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP
1,2,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,13085.0,United Kingdom,HOME,Home decoration,Interior,Pink Cherry Lights,21.29,11.43,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP
2,3,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,13085.0,United Kingdom,HOME,Home decoration,Interior,White Cherry Lights,7.2,2.6,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP
3,4,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,13085.0,United Kingdom,HOME,Home decoration,Interior,"Record Frame 7"" Single Size",21.15,11.45,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP
4,5,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,13085.0,United Kingdom,OTHER,Other,Other,Strawberry Ceramic Trinket Box,12.74,5.85,Regular,Small business,Bronze,Online,Card,WH-UK-01,NONE,0.0,Completed,11,Europe,Western Europe,GBP


In [21]:
import duckdb

duckdb.register('retail_raw', df)

In [22]:
duckdb.sql("""
    SELECT COUNT(*) AS nb_lignes
    FROM retail_raw
""").df()

,nb_lignes
0,1072707


In [23]:
# Décrire les structures de la table: liste des colonnes et leur type actuel (VARCHAR pour toutes, car chargées en mode textepour l'audit)
# VARCHAR veut dire que la donnée est stockée comme du texte, même si elle ressemble à un nombre ou une date.
duckdb.sql("""
    DESCRIBE retail_raw
""").df()

,column_name,column_type,null,key,default,extra
0,OrderLineID,VARCHAR,YES,None,None,None
1,Invoice,VARCHAR,YES,None,None,None
2,StockCode,VARCHAR,YES,None,None,None
3,Description,VARCHAR,YES,None,None,None
4,Quantity,VARCHAR,YES,None,None,None
5,InvoiceDate,VARCHAR,YES,None,None,None
6,CustomerID,VARCHAR,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None
8,ProductCategoryCode,VARCHAR,YES,None,None,None
9,ProductCategory,VARCHAR,YES,None,None,None


Audit — Structure des colonnes (DESCRIBE)

Résultat : La table `retail_raw` contient 27 colonnes, toutes actuellement typées en `VARCHAR` (texte), car le chargement initial a volontairement conservé tout en texte brut afin de permettre un audit sans perte ni erreur de conversion.

Colonnes identifiées, par catégorie :
- Identifiants : `OrderLineID`, `Invoice`, `CustomerID`
- Produit : `StockCode`, `Description`, `ProductCategoryCode`, `ProductCategory`, `ProductFamily`, `ProductNameClean`
- Transaction : `Quantity`, `InvoiceDate`, `UnitPriceGBP`, `UnitCostGBP`
- Client : `Country`, `CustomerSegment`, `CustomerType`, `LoyaltyProgram`
- Vente : `SalesChannel`, `PaymentMethod`, `WarehouseID`, `PromotionCode`



In [34]:
# Afficher Invoice, InvoiceDate, SalesChannel et PaymentMethod pour explorer les données
duckdb.sql("""
    SELECT Invoice, InvoiceDate, SalesChannel, PaymentMethod
    FROM retail_raw
    LIMIT 50
""").df()

,Invoice,InvoiceDate,SalesChannel,PaymentMethod
0,489434,2009-12-01 07:45:00,Online,Card
1,489434,2009-12-01 07:45:00,Online,Card
2,489434,2009-12-01 07:45:00,Online,Card
3,489434,2009-12-01 07:45:00,Online,Card
4,489434,2009-12-01 07:45:00,Online,Card
5,489434,2009-12-01 07:45:00,Online,Card
6,489434,2009-12-01 07:45:00,Online,Card
7,489434,2009-12-01 07:45:00,Online,Card
8,489435,2009-12-01 07:46:00,Phone,Invoice
9,489435,2009-12-01 07:46:00,Phone,Invoice


In [35]:
# Voir les différentes valeurs de SalesChannel et leur fréquence
duckdb.sql("""
    SELECT SalesChannel, COUNT(*) AS nb
    FROM retail_raw
    GROUP BY SalesChannel
    ORDER BY nb DESC
""").df()

,SalesChannel,nb
0,Marketplace,271975
1,Online,271479
2,Store,264177
3,Phone,260881
4,None,4195


In [42]:
# Compter combien de lignes ont SalesChannel = 'NONE'
duckdb.sql("""
    SELECT COUNT(*) AS nb_none
    FROM retail_raw
    WHERE SalesChannel = 'None'
""").df()

,nb_none
0,0


In [43]:
# Confirmer que ce sont bien des vraies cases vides (NULL) et pas le mot "None"
duckdb.sql("""
    SELECT COUNT(*) AS nb_vraiment_vide
    FROM retail_raw
    WHERE SalesChannel IS NULL
""").df()

,nb_vraiment_vide
0,4195


In [36]:
# Voir les différentes valeurs de PaymentMethod et leur fréquence
duckdb.sql("""
    SELECT PaymentMethod, COUNT(*) AS nb
    FROM retail_raw
    GROUP BY PaymentMethod
    ORDER BY nb DESC
""").df()

,PaymentMethod,nb
0,PayPal,273029
1,Card,272611
2,Bank transfer,265183
3,Invoice,261884


In [ ]:
Conclusion — Invoice

Le fichier contient 1 072 707 lignes, mais seulement 53 628 factures différentes.

Cela s'explique simplement : une personne peut acheter plusieurs produits en une seule commande. Par exemple, si un client achète 5 produits différents, cela crée 1 seule facture (Invoice), mais 5 lignes dans le fichier (1 ligne par produit).

Aucune ligne n'a de numéro de facture manquant (0 valeur manquante).

In [28]:
# Vérifier les codes promo : quelles valeurs existent et combien de fois chacune apparaît
duckdb.sql("""
    SELECT PromotionCode, COUNT(*) AS nb
    FROM retail_raw
    GROUP BY PromotionCode
    ORDER BY nb DESC
""").df()

,PromotionCode,nb
0,SPRING,222852
1,NONE,214157
2,WELCOME10,213557
3,XMAS,212674
4,CLEARANCE,207690
5,None,1777


In [47]:
duckdb.sql("""
    SELECT 
        CASE WHEN PromotionCode = 'NONE' OR PromotionCode IS NULL THEN 'NONE' ELSE PromotionCode END AS PromotionCode_clean,
        COUNT(*) AS nb
    FROM retail_raw
    GROUP BY PromotionCode_clean
    ORDER BY nb DESC
""").df()

,PromotionCode_clean,nb
0,SPRING,222852
1,NONE,215934
2,WELCOME10,213557
3,XMAS,212674
4,CLEARANCE,207690


Regrouper les deux colonnes none dans une seule afin que ça soit plus clair 

In [50]:
# Vérifier si Invoice contient uniquement des chiffres, ou s'il y a d'autres caractères (lettres, symboles...)
duckdb.sql("""
    SELECT 
        COUNT(*) AS nb_lignes,
        COUNT(*) - COUNT(TRY_CAST(Invoice AS INTEGER)) AS invoice_non_numerique
    FROM retail_raw
""").df()

,nb_lignes,invoice_non_numerique
0,1072707,19606


In [51]:
# Vérifier si les Invoice non numériques commencent bien par "C"
duckdb.sql("""
    SELECT 
        LEFT(Invoice, 1) AS premiere_lettre,
        COUNT(*) AS nb
    FROM retail_raw
    WHERE TRY_CAST(Invoice AS INTEGER) IS NULL
    GROUP BY premiere_lettre
    ORDER BY nb DESC
""").df()

,premiere_lettre,nb
0,C,19600
1,A,6


In [67]:
# Créer une nouvelle table propre avec Invoice nettoyé (uniquement des chiffres)
duckdb.sql("""
    CREATE OR REPLACE TABLE retail_clean AS
    SELECT 
        * EXCLUDE (Invoice),
        REGEXP_REPLACE(Invoice, '[^0-9]', '', 'g') AS Invoice
    FROM retail_raw
""")

In [68]:
# Vérifier qu'il n'y a plus aucune lettre dans Invoice
duckdb.sql("""
    SELECT 
        COUNT(*) AS nb_lignes,
        COUNT(*) - COUNT(TRY_CAST(Invoice AS INTEGER)) AS invoice_non_numerique
    FROM retail_clean
""").df()

,nb_lignes,invoice_non_numerique
0,1072707,0


Le nettoyage a bien été efectuer et toute les commandes sont en chiffre.

In [69]:
# Harmoniser InvoiceDate : convertir les 2 formats différents en un seul vrai format de date
duckdb.sql("""
    CREATE OR REPLACE TABLE retail_clean AS
    SELECT 
        * EXCLUDE (InvoiceDate),
        CASE 
            WHEN InvoiceDate LIKE '%/%' THEN STRPTIME(InvoiceDate, '%d/%m/%Y %H:%M')
            ELSE STRPTIME(InvoiceDate, '%Y-%m-%d %H:%M:%S')
        END AS InvoiceDate
    FROM retail_clean
""")

In [74]:
# Afficher la colonne InvoiceDate
duckdb.sql("""
    SELECT InvoiceDate
    FROM retail_clean
    LIMIT 10
""").df()

,InvoiceDate
0,2009-12-01 07:45:00
1,2009-12-01 07:45:00
2,2009-12-01 07:45:00
3,2009-12-01 07:45:00
4,2009-12-01 07:45:00
5,2009-12-01 07:45:00
6,2009-12-01 07:45:00
7,2009-12-01 07:45:00
8,2009-12-01 07:46:00
9,2009-12-01 07:46:00


La colonne invoicedate a bien été corrigé et le format date est unifié ainsi que les heures

In [82]:
# Créer LA table finale avec toutes les colonnes nettoyées réunies (Invoice, InvoiceDate, PromotionCode)
duckdb.sql("""
    CREATE OR REPLACE TABLE d_facture_clean AS
    SELECT 
        * EXCLUDE (PromotionCode),
        CASE 
            WHEN PromotionCode = 'NONE' OR PromotionCode IS NULL THEN 'NONE'
            ELSE PromotionCode
        END AS PromotionCode
    FROM retail_clean
""")

In [83]:
# Vérifier que facture_clean contient bien toutes les colonnes nettoyées
duckdb.sql("""
    SELECT Invoice, InvoiceDate, PromotionCode, SalesChannel, PaymentMethod
    FROM facture_clean
    LIMIT 100
""").df()

,Invoice,InvoiceDate,PromotionCode,SalesChannel,PaymentMethod
0,489434,2009-12-01 07:45:00,NONE,Online,Card
1,489434,2009-12-01 07:45:00,NONE,Online,Card
2,489434,2009-12-01 07:45:00,NONE,Online,Card
3,489434,2009-12-01 07:45:00,NONE,Online,Card
4,489434,2009-12-01 07:45:00,NONE,Online,Card
...,...,...,...,...,...
95,489441,2009-12-01 09:44:00,WELCOME10,Store,Bank transfer
96,489442,2009-12-01 09:46:00,WELCOME10,Online,Card
97,489442,2009-12-01 09:46:00,WELCOME10,Online,Card
98,489442,2009-12-01 09:46:00,WELCOME10,Online,Card


Créer une table facture_clean où ce trouve toutes les colonnes modifier et clean

In [84]:
duckdb.sql("""
    COPY d_facture_clean TO 'export_final.parquet' (FORMAT PARQUET)
""")